In [ ]:
from pathlib import Path
import json

from PIL import Image, ImageOps
from torchvision import transforms

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

split_manifest = json.loads(
    (PROJECT_ROOT / "reports" / "base_split.json").read_text(
        encoding="utf-8"
    )
)
print("Sources:", split_manifest["sources"])
print("Split:", len(split_manifest["training_paths"]), len(split_manifest["validation_paths"]))

IMAGE_SIZE = 128

def resize_with_padding(image):
    return ImageOps.pad(
        image.convert("RGB"),
        (IMAGE_SIZE, IMAGE_SIZE),
        method=Image.Resampling.BILINEAR,
        color=(0, 0, 0),
    )

base_transform = transforms.Compose([
    transforms.Lambda(resize_with_padding),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])

sample_path = PROJECT_ROOT / split_manifest["training_paths"][0]

with Image.open(sample_path) as image:
    sample_tensor = base_transform(image)

print("Input tensor shape:", sample_tensor.shape)
print("Pixel range:", sample_tensor.min().item(), sample_tensor.max().item())

In [ ]:
with Image.open(sample_path) as image:
    prepared = resize_with_padding(image)

    print("Original:", image.size)
    print("Prepared:", prepared.size)
    display(prepared)

In [ ]:
from torch.utils.data import Dataset

class ManifestImageDataset(Dataset):
    """One class mapping and approved sample list across all source folders."""
    def __init__(self, manifest, transform=None):
        self.transform = transform
        self.class_to_idx = manifest["class_to_idx"]
        self.classes = sorted(self.class_to_idx, key=self.class_to_idx.get)
        self.samples = [
            (str(PROJECT_ROOT / row["path"]), self.class_to_idx[row["label"]])
            for row in manifest["samples"]
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_dataset = ManifestImageDataset(split_manifest)
print("Approved images:", len(train_dataset))
print("Class mapping:", train_dataset.class_to_idx)

In [ ]:
path_to_index = {
    Path(path).relative_to(PROJECT_ROOT).as_posix(): index
    for index, (path, label) in enumerate(train_dataset.samples)
}
train_indices = [path_to_index[path] for path in split_manifest["training_paths"]]
val_indices = [path_to_index[path] for path in split_manifest["validation_paths"]]
assert train_indices and val_indices
assert not set(train_indices) & set(val_indices)
assert set(train_indices) | set(val_indices) == set(range(len(train_dataset)))
print("Training indices:", len(train_indices))
print("Validation indices:", len(val_indices))

In [ ]:
from torch.utils.data import Subset

# Separate transforms can be applied later to this same approved sample list.
train_full = ManifestImageDataset(split_manifest, transform=base_transform)
validation_full = ManifestImageDataset(split_manifest, transform=base_transform)
assert train_dataset.samples == train_full.samples == validation_full.samples
train_subset = Subset(train_full, train_indices)
validation_subset = Subset(validation_full, val_indices)
print("Training images:", len(train_subset))
print("Validation images:", len(validation_subset))
print("Classes:", train_full.classes)
image_tensor, label_index = train_subset[0]
print("One image shape:", image_tensor.shape)
print("Its class:", train_full.classes[label_index])

In [ ]:
import torch
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

batch_images, batch_labels = next(iter(train_loader))

print("Images shape:", batch_images.shape)
print("Labels shape:", batch_labels.shape)
print("First five labels:", batch_labels[:5])


In [ ]:
from torch import nn

# Conv2d output per spatial axis (dilation=1): floor((N + 2*P - K) / S) + 1.
# Here N=128, P=1, K=3, S=1: floor((128 + 2*1 - 3) / 1) + 1 = 128.
# 16 filters make 16 feature maps of 128x128; MaxPool2d(2) makes them 64x64.
first_block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
)

first_output = first_block(batch_images)

print("Input shape:", batch_images.shape)
print("Output shape:", first_output.shape)

In [ ]:
# Here N=64, P=1, K=3, S=1: floor((64 + 2*1 - 3) / 1) + 1 = 64.
# 32 filters make 32 feature maps of 64x64; MaxPool2d(2) makes them 32x32.
second_block = nn.Sequential(
    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
)

second_output = second_block(first_output)

print("First block output:", first_output.shape)
print("Second block output:", second_output.shape)

In [ ]:
flatten = nn.Flatten(start_dim=1)
flat_features = flatten(second_output)

print("Before flatten:", second_output.shape)
print("After flatten:", flat_features.shape)

In [ ]:
num_classes = len(train_full.classes)

classifier = nn.Linear(
    in_features=32 * 32 * 32,
    out_features=num_classes,
)

class_scores = classifier(flat_features)

print("Number of classes:", num_classes)
print("Scores shape:", class_scores.shape)
print("First image scores:", class_scores[0])

In [ ]:
first_image_scores = class_scores[0]

for class_index, score in enumerate(first_image_scores):
    class_name = train_full.classes[class_index]
    print(f"{class_index}: {class_name:10s} | score: {score.item():.4f}")

predicted_index = first_image_scores.argmax().item()
target_index = batch_labels[0].item()

print("Predicted class:", train_full.classes[predicted_index])
print("True class:", train_full.classes[target_index])

In [ ]:
loss_fn = nn.CrossEntropyLoss()
batch_loss = loss_fn(class_scores, batch_labels)

per_image_losses = nn.functional.cross_entropy(
    class_scores,
    batch_labels,
    reduction="none",
)

print("Scores shape:", class_scores.shape)
print("Labels shape:", batch_labels.shape)
print("First image loss:", per_image_losses[0].item())
print("Mean of image losses:", per_image_losses.mean().item())
print("Batch loss:", batch_loss.item())

In [ ]:
layers = {
    "first conv": first_block[0],
    "second conv": second_block[0],
    "classifier": classifier,
}

for name, layer in layers.items():
    print(name, "gradient before backward:", layer.weight.grad is None)

batch_loss.backward()

for name, layer in layers.items():
    print(
        name,
        "gradient shape:", tuple(layer.weight.grad.shape),
        "mean absolute gradient:", layer.weight.grad.abs().mean().item(),
    )

In [ ]:
all_parameters = [
    *first_block.parameters(),
    *second_block.parameters(),
    *classifier.parameters(),
]

optimizer = torch.optim.SGD(all_parameters, lr=0.01)

old_loss = batch_loss.item()
old_classifier_weights = classifier.weight.detach().clone()

optimizer.step()

with torch.no_grad():
    new_scores = classifier(
        flatten(
            second_block(
                first_block(batch_images)
            )
        )
    )
    new_loss = loss_fn(new_scores, batch_labels)

largest_weight_change = (
    classifier.weight.detach() - old_classifier_weights
).abs().max().item()

print("Old loss:", old_loss)
print("New loss:", new_loss.item())
print("Largest classifier weight change:", largest_weight_change)

In [ ]:
print("Before clearing:", classifier.weight.grad is None)
print("Gradient size:", classifier.weight.grad.abs().mean().item())

optimizer.zero_grad(set_to_none=True)

for name, layer in layers.items():
    print(name, "gradient after clearing:", layer.weight.grad)

In [ ]:
class VehicleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(32 * 32 * 32, num_classes),
        )

    def forward(self, images):
        features = self.features(images)
        scores = self.classifier(features)
        return scores


torch.manual_seed(42)
model = VehicleCNN(num_classes=len(train_full.classes))

print(model)
print("Output shape:", model(batch_images).shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)

LEARNING_RATE = 1e-3

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Device:", device)
print("Learning rate:", LEARNING_RATE)
print("Optimizer:", type(optimizer).__name__)

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        scores = model(images)
        loss = loss_fn(scores, labels)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size

        predicted_indices = scores.argmax(dim=1)
        correct_predictions += (
            predicted_indices == labels
        ).sum().item()

        total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }

In [ ]:
def evaluate(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            scores = model(images)
            loss = loss_fn(scores, labels)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size

            predicted_indices = scores.argmax(dim=1)
            correct_predictions += (
                predicted_indices == labels
            ).sum().item()

            total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


In [ ]:
initial_validation = evaluate(
    model,
    validation_loader,
    loss_fn,
    device,
)

print("Initial validation loss:", initial_validation["loss"])
print(
    "Initial validation accuracy:",
    f'{initial_validation["accuracy"]:.1%}',
)

In [ ]:
from copy import deepcopy

# Start this experiment with fresh weights and a fixed shuffle seed.
torch.manual_seed(42)
train_loader.generator.manual_seed(42)

model = VehicleCNN(
    num_classes=len(train_full.classes)
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

EPOCHS = 10
history = []

best_val_accuracy = -1.0
best_epoch = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, loss_fn, device
    )
    val_metrics = evaluate(
        model, validation_loader, loss_fn, device
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    })

    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss: {train_metrics["loss"]:.4f} | '
        f'train acc: {train_metrics["accuracy"]:.1%} | '
        f'val loss: {val_metrics["loss"]:.4f} | '
        f'val acc: {val_metrics["accuracy"]:.1%}'
    )

print(
    f"Best validation accuracy: {best_val_accuracy:.1%} "
    f"at epoch {best_epoch}"
)

> اعداد این بخش از اجرای قبلی روی ۳۲۰ تصویر آموزش و ۸۰ تصویر validation هستند؛ نتیجهٔ تقسیم مشترک فعلی نیستند. پس از آموزش مجدد، همین بخش را با نتایج جدید به‌روز می‌کنیم.

## CNN baseline — results and interpretation

### Experiment configuration

The baseline was trained from scratch on 320 images and validated on 80 images, with 40 training and 10 validation images per class. The eight-class split and shuffle seed were fixed at 42. Images were prepared with aspect-ratio-preserving padding to 128 × 128, converted to tensors, and normalized with mean and standard deviation 0.5 for each RGB channel.

The architecture contains two Conv2d–ReLU–MaxPool2d blocks (16 and 32 output channels), followed by Flatten and an eight-class Linear head. It has **267,240 trainable parameters**. Training used CrossEntropyLoss, Adam with learning rate **0.001**, batch size **32**, and **10 epochs**. This reference run used no random augmentation, dropout, weight decay, or learning-rate scheduler.

### Recorded results

Before training, validation accuracy was **13.8% (11/80)** and validation loss was **2.0716**. A uniform eight-class prediction would have cross-entropy ln(8) ≈ 2.0794 and an expected accuracy of 12.5%, so this initial result is consistent with an untrained reference.

| Epoch | Training loss | Training accuracy | Validation loss | Validation accuracy | Observation |
|---:|---:|---:|---:|---:|---|
| 1 | 2.2503 | 26.6% | 2.0185 | 25.0% | Early learning |
| 2 | 1.6129 | 47.2% | **1.7916** | 37.5% | Lowest validation loss |
| 7 | 0.2736 | 95.0% | 2.2770 | **52.5%** | Highest validation accuracy |
| 10 | 0.0745 | 99.7% | 2.5228 | 48.8% | Final epoch |

The checkpoint selected by the current **validation-accuracy criterion is epoch 7**, with **42 correct predictions out of 80**. The final epoch classified **39/80** validation images correctly. The best checkpoint is currently held in `best_state` in notebook memory; it has not yet been saved to disk or loaded back into `model`. The current `model` still contains the epoch-10 weights.

### Interpretation

The model learned the training data well: reported training loss decreased from 2.2503 to 0.0745, and training accuracy increased from 26.6% to 99.7%. Validation improvement was much smaller. After reaching its minimum at epoch 2, validation loss increased overall to 2.5228, while validation accuracy fluctuated and ended below its epoch-7 maximum. This divergence is strong evidence of **overfitting**: better fitting of the training examples did not translate into comparable generalization to the held-out validation images.

The final reported train–validation accuracy difference is about **50.9 percentage points**. Training metrics are accumulated while weights change within an epoch, whereas validation metrics are measured after the epoch. For an exact comparison at a fixed checkpoint, both splits should be evaluated using `evaluate` after restoring that checkpoint.

Validation accuracy and cross-entropy do not measure the same thing. Accuracy checks whether the highest-scoring class is correct; cross-entropy also depends on the probability assigned to the true class. Thus accuracy can improve while loss rises if some errors become more confident. This is a possible explanation of the observed curves, not yet a confirmed diagnosis; confidence and per-image errors still need inspection. Selection by validation loss would choose epoch 2 rather than epoch 7, so the checkpoint-selection criterion must be reported explicitly.

The Linear head contains **262,152 parameters**, about **98.1%** of the model's parameters. This large head relative to only 320 training images is a plausible contributor to overfitting, but the run alone does not isolate its causal effect. Controlled changes are needed before attributing the result to a specific architectural or training choice.

With 80 validation images, one additional correct prediction changes accuracy by **1.25 percentage points**. Small differences between runs should therefore be interpreted cautiously rather than treated as decisive evidence of improvement.

### Required follow-up

- Save the epoch-7 checkpoint with class mapping, preprocessing, seed, and run configuration; plot the complete training and validation curves.
- Evaluate that checkpoint for macro precision, macro recall, macro-F1, per-class metrics, and confusion matrices; identify the lowest-precision and lowest-recall classes and inspect errors.
- Compare augmentation, dropout, pooling, regularization, scheduling, sampling, and loss choices through the controlled experiments required by the project brief, keeping other factors fixed.

**The reserved test split has not been evaluated.** Validation guides experiment comparison; the frozen test set is reserved for one final evaluation after choosing the configuration. This baseline establishes a measured reference, not the final model or a final test result.


---

## تحلیل فارسی مدل پایه

### تنظیمات آزمایش

مدل پایه از صفر، با ۳۲۰ تصویر آموزشی و ۸۰ تصویر validation آموزش داده شد. در هر یک از ۸ کلاس، ۴۰ تصویر برای آموزش و ۱۰ تصویر برای validation داریم. بذر تصادفی تقسیم‌بندی و ترتیب batchها برابر ۴۲ است. تصاویر با حفظ نسبت طول و عرض و افزودن حاشیه به اندازهٔ ۱۲۸×۱۲۸ آماده شدند. سپس به تنسور تبدیل شدند و برای هر کانال RGB با میانگین و انحراف معیار ۰٫۵ نرمال‌سازی شدند.

مدل دو بلوک Conv2d، ReLU و MaxPool2d با ۱۶ و ۳۲ کانال خروجی دارد. بعد از آن Flatten و یک لایهٔ Linear با ۸ خروجی قرار دارد. تعداد پارامترهای قابل‌آموزش ۲۶۷٬۲۴۰ است. آموزش با Adam، نرخ یادگیری ۰٫۰۰۱، CrossEntropyLoss، batchهای ۳۲تایی و ۱۰ epoch انجام شد. در این آزمایش augmentation تصادفی، dropout، weight decay و scheduler استفاده نشدند تا نتیجهٔ مرجع داشته باشیم.

### نتیجه‌های اصلی

قبل از آموزش، دقت validation برابر ۱۳٫۸٪، یعنی ۱۱ پاسخ درست از ۸۰ تصویر، و خطای آن برابر ۲٫۰۷۱۶ بود. این نتیجه به مرجعِ پیش‌بینی تقریباً یکنواخت بین ۸ کلاس نزدیک است: دقت مورد انتظارِ حدس تصادفی ۱۲٫۵٪ و خطای پیش‌بینی یکنواخت تقریباً ۲٫۰۷۹۴ است.

- بهترین دقت validation در epoch هفتم برابر **۵۲٫۵٪** بود؛ یعنی **۴۲ پاسخ درست از ۸۰ تصویر**.
- کمترین خطای validation در epoch دوم برابر **۱٫۷۹۱۶** بود؛ این epoch لزوماً بیشترین accuracy را ندارد.
- در epoch دهم، دقت آموزشی به **۹۹٫۷٪** و خطای آموزشی به **۰٫۰۷۴۵** رسید؛ اما دقت validation برابر **۴۸٫۸٪**، یعنی ۳۹ پاسخ درست از ۸۰ تصویر، و خطای آن برابر **۲٫۵۲۲۸** بود.
- معیار انتخاب checkpoint در این اجرا، بیشترین **accuracy روی validation** است؛ بنابراین وزن‌های epoch هفتم انتخاب شده‌اند، نه وزن‌های epoch آخر.

### چرا نتیجه نشانهٔ overfitting است؟

خطای آموزش از ۲٫۲۵۰۳ در epoch اول به ۰٫۰۷۴۵ در epoch آخر کاهش یافته است. مدل روی تصاویر آموزشی بسیار بهتر شده، اما پس از epoch دوم، خطای validation در مجموع افزایش یافته و دقت آن هم‌پای دقت آموزشی رشد نکرده است. این اختلاف نشان می‌دهد یادگیریِ نمونه‌های آموزشی، به همان اندازه به تشخیص تصاویر جداگذاشته‌شده تعمیم پیدا نکرده است؛ این روند شواهد قویِ overfitting دارد.

فاصلهٔ دقت‌های گزارش‌شده در epoch آخر حدود ۵۰٫۹ **واحد درصد** است. با این حال، دقت آموزش هنگام تغییر وزن‌ها در طول epoch جمع‌آوری می‌شود، ولی validation با وزن‌های ثابتِ پایان epoch محاسبه می‌شود. برای مقایسهٔ دقیق روی یک checkpoint ثابت، باید هر دو مجموعه را بعد از بارگذاری همان checkpoint با تابع evaluate ارزیابی کنیم.

### چرا ممکن است accuracy بیشتر شود، ولی loss هم افزایش پیدا کند؟

Accuracy فقط می‌پرسد آیا کلاسِ دارای بیشترین نمره درست است یا نه. CrossEntropyLoss به احتمال اختصاص‌یافته به کلاس واقعی هم حساس است. بنابراین ممکن است تعداد پاسخ‌های درست بیشتر شود، ولی بعضی پاسخ‌های اشتباه با اطمینان بیشتری داده شوند و خطا را بالا ببرند. این توضیح برای منحنی‌های فعلی ممکن است، اما تا قبل از بررسی احتمال‌ها و خطای تک‌تک تصاویر، آن را علتِ اثبات‌شده نمی‌دانیم.

اگر معیار انتخاب مدل کمترین validation loss بود، epoch دوم انتخاب می‌شد؛ چون معیار فعلی بیشترین validation accuracy است، epoch هفتم انتخاب شده است. پس باید معیار انتخاب را همراه نتیجه گزارش کنیم.

### نقش اندازهٔ مدل و تعداد داده‌ها

لایهٔ Linear آخر ۲۶۲٬۱۵۲ پارامتر، یعنی حدود ۹۸٫۱٪ پارامترهای کل مدل، را دارد. این لایهٔ بزرگ در برابر فقط ۳۲۰ تصویر آموزشی، می‌تواند به overfitting کمک کند؛ اما برای اثبات اثر آن باید یک آزمایش کنترل‌شده انجام دهیم و سایر عوامل را ثابت نگه داریم.

مجموعهٔ validation فقط ۸۰ تصویر دارد؛ بنابراین درست‌شدنِ یک تصویر بیشتر، accuracy را ۱٫۲۵ واحد درصد افزایش می‌دهد. اختلاف‌های کوچکِ دقت را نباید به‌تنهایی بهبود قطعی بدانیم.

### ادامهٔ کار و حدود نتیجه‌گیری

۱. بهترین checkpoint را همراه نگاشت کلاس‌ها، پیش‌پردازش، بذر تصادفی و تنظیمات آزمایش ذخیره کنیم و نمودارهای کامل آموزش و validation را رسم کنیم.
۲. برای checkpoint منتخب، precision، recall و F1 هر کلاس، معیارهای macro و confusion matrix را حساب کنیم؛ سپس کلاس‌های دارای کمترین precision و recall و تصاویر اشتباه را بررسی کنیم.
۳. آزمایش‌های کنترل‌شدهٔ خواسته‌شده در صورت‌سؤال، از جمله augmentation، dropout، pooling، weight decay، scheduler، balanced sampling و CE/BCE را انجام دهیم.
۵. تصاویر واجد شرایط از unclean را پس از بررسی کیفیت و برچسب و کنترل تکرار با validation و test، در یک آزمایش جداگانه به آموزش اضافه کنیم. validation فعلی برای مقایسهٔ اثر این داده‌ها ثابت می‌ماند. داده‌های neysan و نمونهٔ دارای تعارض vanet/neysan جداگانه تعیین تکلیف می‌شوند.

**این نتیجه فقط baseline با train اصلی است و نتیجهٔ مدل نهایی نیست. Test هنوز ارزیابی نشده است.** تصمیم‌های آزمایش را بر اساس validation می‌گیریم و test کنارگذاشته‌شده را پس از انتخاب تنظیمات نهایی، یک بار ارزیابی می‌کنیم.

در زمان ثبت این تحلیل، best_state در حافظهٔ نوت‌بوک وزن‌های epoch هفتم را نگه می‌دارد و model هنوز وزن‌های epoch دهم را دارد. ذخیرهٔ checkpoint روی دیسک، قدم بعدی است.


In [ ]:
checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "cnn_baseline_best.pt"

checkpoint = {
    "architecture": "VehicleCNN",
    "model_state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in best_state.items()
    },
    "class_to_idx": train_full.class_to_idx,
    "best_epoch": best_epoch,
    "best_val_accuracy": best_val_accuracy,
    "preprocessing": {
        "image_size": IMAGE_SIZE,
        "resize": "aspect-ratio-preserving padding",
        "interpolation": "bilinear",
        "padding_color": [0, 0, 0],
        "mean": [0.5, 0.5, 0.5],
        "std": [0.5, 0.5, 0.5],
    },
    "training_config": {
        "seed": 42,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "loss": "CrossEntropyLoss",
    },
    "split_manifest": split_manifest,
    "history": history,
    "review_threshold": None,  # Not calibrated yet.
}

# This is a binary file; do not edit or resave it as text.
torch.save(checkpoint, checkpoint_path)

# Verify that the saved checkpoint can be read immediately.
verified_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)
assert verified_checkpoint["best_epoch"] == best_epoch
assert verified_checkpoint["class_to_idx"] == train_full.class_to_idx
print("Checkpoint read-back verified.")

print("Saved to:", checkpoint_path)
print("Best epoch:", checkpoint["best_epoch"])
print("Validation accuracy:", f'{best_val_accuracy:.1%}')

In [ ]:
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in history]
train_losses = [row["train_loss"] for row in history]
val_losses = [row["val_loss"] for row in history]

train_accuracies = [
    100 * row["train_accuracy"] for row in history
]
val_accuracies = [
    100 * row["val_accuracy"] for row in history
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, marker="o", label="Train")
axes[0].plot(epochs, val_losses, marker="o", label="Validation")
axes[0].set_title("Cross-entropy loss")
axes[0].set_ylabel("Loss")

axes[1].plot(epochs, train_accuracies, marker="o", label="Train")
axes[1].plot(epochs, val_accuracies, marker="o", label="Validation")
axes[1].set_title("Classification accuracy")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(0, 100)

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_xticks(epochs)
    ax.axvline(
        best_epoch,
        linestyle="--",
        color="gray",
        label="Selected epoch",
    )
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()

figure_dir = PROJECT_ROOT / "reports" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    figure_dir / "cnn_baseline_curves.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

In [ ]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "cnn_baseline_best.pt"

# Recover a checkpoint accidentally resaved as UTF-16 text.
with checkpoint_path.open("rb") as checkpoint_file:
    header = checkpoint_file.read(2)

if header in (b"\xfe\xff", b"\xff\xfe"):
    if "checkpoint" not in globals():
        raise RuntimeError(
            "The checkpoint was converted to text. "
            "Run the checkpoint save cell using best_state still in memory."
        )
    assert checkpoint["class_to_idx"] == train_full.class_to_idx
    torch.save(checkpoint, checkpoint_path)
    print("Restored binary checkpoint from the in-memory checkpoint dictionary.")

saved_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

assert (
    saved_checkpoint["class_to_idx"]
    == train_full.class_to_idx
)

if saved_checkpoint["split_manifest"].get("fingerprint") != split_manifest["fingerprint"]:
    raise RuntimeError("Checkpoint belongs to a different data split. Train and save with the current split first.")

best_model = VehicleCNN(
    num_classes=len(saved_checkpoint["class_to_idx"])
).to(device)

best_model.load_state_dict(
    saved_checkpoint["model_state_dict"]
)

best_validation = evaluate(
    best_model,
    validation_loader,
    loss_fn,
    device,
)

print("Loaded epoch:", saved_checkpoint["best_epoch"])
print("Validation loss:", best_validation["loss"])
print("Validation accuracy:", f'{best_validation["accuracy"]:.1%}')

In [ ]:
best_model.eval()

y_true = []
y_pred = []
probability_batches = []

with torch.no_grad():
    for images, labels in validation_loader:
        scores = best_model(images.to(device))
        probabilities = torch.softmax(scores, dim=1)
        predictions = scores.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())
        probability_batches.append(probabilities.cpu())

val_probabilities = torch.cat(probability_batches, dim=0)

correct = sum(
    true_label == predicted_label
    for true_label, predicted_label in zip(y_true, y_pred)
)

print("Number of images:", len(y_true))
print("Probability shape:", val_probabilities.shape)
print("Correct predictions:", correct)
print("First ten true labels:", y_true[:10])
print("First ten predictions:", y_pred[:10])

In [ ]:
from sklearn.metrics import classification_report

class_names = train_full.classes
class_indices = list(range(len(class_names)))

report_text = classification_report(
    y_true,
    y_pred,
    labels=class_indices,
    target_names=class_names,
    digits=3,
    zero_division=0,
)

print(report_text)

baseline_report = classification_report(
    y_true,
    y_pred,
    labels=class_indices,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

baseline_report["data_split"] = {"fingerprint": split_manifest["fingerprint"], "training_images": len(train_subset), "validation_images": len(validation_subset)}

report_path = PROJECT_ROOT / "reports" / "cnn_baseline_validation_metrics.json"
report_path.write_text(
    json.dumps(baseline_report, indent=2),
    encoding="utf-8",
)

> اعداد این بخش از اجرای قبلی روی ۳۲۰ تصویر آموزش و ۸۰ تصویر validation هستند؛ نتیجهٔ تقسیم مشترک فعلی نیستند. پس از آموزش مجدد، همین بخش را با نتایج جدید به‌روز می‌کنیم.

## Baseline validation — per-class findings / تحلیل معیارهای هر کلاس

The epoch-7 checkpoint achieves accuracy **52.5%**, macro precision **56.31%**, macro recall **52.50%**, and macro-F1 **50.23%** on 80 validation images. Each class has support 10; therefore macro and support-weighted averages are equal in this split.

- **Lowest precision: ambulance (41.67%).** All 10 true ambulances were found (recall 100%), but the model predicted ambulance for 24 images, including 14 false positives. High recall alone therefore does not establish reliable ambulance predictions.
- **Lowest recall: minibus (20%).** Only 2 of 10 minibuses were correctly identified; 8 were assigned other labels. The model predicted minibus for 3 images, of which 2 were correct (precision 66.67%). Its F1 is also the lowest, at 30.77%.
- Vanet recall is 30% (3/10), so this class also needs error inspection. Savari has the highest per-class F1 (66.67%) in this run.

These counts describe the errors, but do not establish their causes. Inspect the confusion matrices and actual images before attributing mistakes to visual similarity, lighting, cropping, or label ambiguity.

### توضیح فارسی

این ارزیابی مربوط به وزن‌های epoch هفتم است، نه epoch آخر. دقت کلی ۵۲٫۵٪ است؛ macro precision برابر ۵۶٫۳۱٪، macro recall برابر ۵۲٫۵٪ و macro-F1 برابر ۵۰٫۲۳٪ است. چون هر کلاس ۱۰ تصویر واقعی دارد، میانگین macro و weighted در این مجموعه برابر شده‌اند. Macro-F1 میانگین F1 هشت کلاس است؛ نباید آن را از میانگین precision و recall دوباره محاسبه کنیم.

**پاسخ سؤال استاد دربارهٔ کمترین precision:** کلاس ambulance با precision برابر ۴۱٫۶۷٪ ضعیف‌ترین است. مدل هر ۱۰ آمبولانس واقعی را پیدا کرده، بنابراین recall آن ۱۰۰٪ است. ولی در مجموع ۲۴ تصویر را آمبولانس پیش‌بینی کرده و ۱۴ مورد از آن‌ها متعلق به کلاس‌های دیگر بوده‌اند. بنابراین «همهٔ آمبولانس‌ها را پیدا کرده» با «هر وقت می‌گوید آمبولانس، درست می‌گوید» یکسان نیست.

**پاسخ سؤال استاد دربارهٔ کمترین recall:** کلاس minibus با recall برابر ۲۰٪ ضعیف‌ترین است. از ۱۰ مینی‌بوس واقعی فقط ۲ مورد درست تشخیص داده شده و ۸ مورد از دست رفته‌اند. مدل فقط ۳ تصویر را مینی‌بوس پیش‌بینی کرده که ۲ موردشان درست بوده‌اند؛ بنابراین precision آن ۶۶٫۶۷٪ است، ولی پوشش این کلاس پایین است. F1 این کلاس نیز با ۳۰٫۷۷٪ کمترین مقدار را دارد.

کلاس vanet هم فقط ۳ تصویر از ۱۰ تصویر واقعی را درست تشخیص داده است و باید خطاهایش بررسی شوند. بالاترین F1 فعلی مربوط به savari با ۶۶٫۶۷٪ است؛ با این حال دقت کلی به‌تنهایی این تفاوت عملکرد کلاس‌ها را نشان نمی‌دهد.

این گزارش هنوز نشان نمی‌دهد آمبولانس‌های اشتباه از کدام کلاس‌ها آمده‌اند یا مینی‌بوس‌ها با چه کلاس‌هایی اشتباه شده‌اند. پاسخ آن‌ها را از confusion matrix و بررسی تصاویر می‌گیریم. علت‌های احتمالی مانند شباهت ظاهری، نور، برش تصویر یا ابهام برچسب، تا پیش از بررسی شواهد علت قطعی محسوب نمی‌شوند.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_counts = confusion_matrix(
    y_true, y_pred, labels=class_indices
)

cm_normalized = confusion_matrix(
    y_true, y_pred, labels=class_indices, normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ConfusionMatrixDisplay(
    cm_counts, display_labels=class_names
).plot(
    ax=axes[0],
    cmap="Blues",
    values_format="d",
    xticks_rotation=45,
    colorbar=False,
)

ConfusionMatrixDisplay(
    cm_normalized, display_labels=class_names
).plot(
    ax=axes[1],
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    colorbar=False,
)

axes[0].set_title("Validation — image counts")
axes[1].set_title("Validation — normalized by true class")

plt.tight_layout()
plt.savefig(
    figure_dir / "cnn_baseline_confusion_matrices.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## ماتریس خطای جداگانه برای هر کلاس

برای هر کلاس، مسئله را موقتاً به «این کلاس» و «سایر کلاس‌ها» تبدیل می‌کنیم. در هر ماتریس ۲×۲، **سطر برچسب واقعی** و **ستون پیش‌بینی** است:

| | پیش‌بینیِ سایر | پیش‌بینیِ کلاس |
|---|---:|---:|
| واقعیِ سایر | TN | FP |
| واقعیِ کلاس | FN | TP |

برای مثال، تصویری که واقعاً `kamyun` است ولی `kamyunet` پیش‌بینی شده، برای `kamyun` یک FN و برای `kamyunet` یک FP است. ماتریس ۸×۸ قبلی همچنان برای دیدن *جفت کلاس‌های اشتباه‌شده* لازم است؛ این هشت ماتریس برای محاسبهٔ TP/FP/FN/TN هر کلاس‌اند.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for class_index, (class_name, ax) in enumerate(zip(class_names, axes.flat)):
    # One-versus-rest: True means "this class", False means "all other classes".
    actual_is_class = [label == class_index for label in y_true]
    predicted_is_class = [label == class_index for label in y_pred]
    binary_cm = confusion_matrix(
        actual_is_class, predicted_is_class, labels=[False, True]
    )
    tn, fp, fn, tp = binary_cm.ravel()
    assert tn + fp + fn + tp == len(y_true)
    print(f"{class_name:10} | TP={tp:3d} FP={fp:3d} FN={fn:3d} TN={tn:3d}")

    ConfusionMatrixDisplay(
        binary_cm, display_labels=["Other", class_name]
    ).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(class_name)

plt.tight_layout()
plt.show()


## بررسی خطاهای با اطمینان بالا

در ماتریس خطا، سطر برچسب واقعی و ستون پیش‌بینی مدل است. ۱۳ کامیون به کامیونت و ۹ کامیونت به کامیون اشتباه شده‌اند. ۸ کامیونت نیز آمبولانس پیش‌بینی شده‌اند. این اعداد به‌تنهایی علت اشتباه را ثابت نمی‌کنند؛ باید تصاویر را بررسی کنیم.

سلول بعدی تمام موارد اشتباه را پیدا می‌کند، آن‌ها را بر اساس احتمالِ کلاس پیش‌بینی‌شده از زیاد به کم مرتب می‌کند و ۱۲ مورد اول را نشان می‌دهد. این مقدار، اطمینان خروجی softmax است، نه احتمال تضمین‌شدهٔ درست‌بودن پاسخ. اگر برچسبی مشکوک بود، فایل خام را تغییر نده؛ ابتدا آن را در فهرست بازبینی محلی ثبت کنیم.


In [ ]:
# The validation DataLoader uses shuffle=False, so these positions match y_true/y_pred.
wrong_indices = [
    i for i, (true_label, predicted_label) in enumerate(zip(y_true, y_pred))
    if true_label != predicted_label
]
wrong_indices.sort(
    key=lambda i: val_probabilities[i, y_pred[i]].item(),
    reverse=True,
)

print("Validation mistakes:", len(wrong_indices), "out of", len(y_true))
print("Highest-confidence mistakes shown:", min(12, len(wrong_indices)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, position in zip(axes.flat, wrong_indices[:12]):
    # Subset index -> full dataset index -> original image path.
    full_index = validation_subset.indices[position]
    image_path, _ = validation_full.samples[full_index]
    confidence = val_probabilities[position, y_pred[position]].item()

    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(
        f"True: {class_names[y_true[position]]} | "
        f"Pred: {class_names[y_pred[position]]}\n"
        f"Model confidence: {confidence:.1%}", fontsize=9
    )
    ax.axis("off")
for ax in list(axes.flat)[min(12, len(wrong_indices)):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
position = wrong_indices[0]
full_index = validation_subset.indices[position]
image_path, _ = validation_full.samples[full_index]

print("Image:", image_path)
print("True:", class_names[y_true[position]])
print("Predicted:", class_names[y_pred[position]])

with Image.open(image_path) as image:
    display(image.copy())

## کدام دو کلاس بیشتر با هم اشتباه می‌شوند؟

طبق صورت پروژه، برای هر جفت کلاس، خطای دو جهت را از ماتریسِ نرمال‌شده با سطر واقعی جمع می‌کنیم:

```text
pair_confusion(i, j) = C_normalized[i, j] + C_normalized[j, i]
```

عدد اول می‌گوید چه سهمی از تصاویر واقعی کلاس `i` به اشتباه `j` پیش‌بینی شده‌اند؛ عدد دوم جهت عکس را نشان می‌دهد. این *امتیاز رتبه‌بندیِ جفت‌ها* است، نه درصد دقت مدل یا احتمال یک تصویر. بالا بودنش دلیل کافی برای ادغام کلاس‌ها یا جابه‌جایی فایل‌ها نیست؛ باید تصویر، تعریف کلاس و هزینهٔ خطا را هم بررسی کنیم.


In [ ]:
pair_results = []
for first in range(len(class_names)):
    for second in range(first + 1, len(class_names)):
        first_to_second = cm_normalized[first, second]
        second_to_first = cm_normalized[second, first]
        pair_results.append((
            first_to_second + second_to_first,
            first, second,
            first_to_second, second_to_first,
        ))

pair_results.sort(reverse=True)
for rank, (score, first, second, first_to_second, second_to_first) in enumerate(pair_results[:5], start=1):
    print(
        f"{rank}. {class_names[first]} ↔ {class_names[second]} | "
        f"{class_names[first]}→{class_names[second]}: "
        f"{cm_counts[first, second]}/{cm_counts[first].sum()} = {first_to_second:.3f} | "
        f"{class_names[second]}→{class_names[first]}: "
        f"{cm_counts[second, first]}/{cm_counts[second].sum()} = {second_to_first:.3f} | "
        f"pair score: {score:.3f}"
    )


## آزمایش کنترل‌شدهٔ افزودن داده‌های unclean

مدل قبلی فقط داده‌های تأییدشدهٔ train و test قدیمی را دید. در این آزمایش، ۱۷۵۵ تصویر یکتای `unclean` با یکی از هشت برچسب شناخته‌شده را **با برچسب فعلی و به‌صورت موقت** فقط به آموزش اضافه می‌کنیم. دو تکرار دقیقِ validation، ۲۱ تکرار دقیقِ آموزش و تعارض برچسب کنار گذاشته می‌شوند. `neysan` وارد این آزمایش نمی‌شود. هیچ فایل خامی جابه‌جا یا بازبرچسب‌گذاری نمی‌شود.

هدف این است که روی همان ۴۶۷ تصویر validation، اثر افزودن دادهٔ بالقوه نویزی را بسنجیم. تنظیمات CNN، seed، batch size و تعداد epoch مثل اجرای قبلی‌اند. چون دادهٔ آموزش بیشتر شده، هر epoch شامل قدم‌های بهینه‌سازی بیشتری می‌شود؛ پس اختلاف نتیجه را نباید فقط به «بهتر بودن برچسب‌ها» نسبت داد. کد محاسبات کمکی در `scripts/run_experiments.py` است؛ سلول‌های زیر خودِ انتخاب داده، ساخت مدل، حلقهٔ آموزش و ذخیرهٔ نتیجه را نشان می‌دهند.

In [ ]:
import sys
from pathlib import Path
from collections import Counter
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.run_experiments import (
    manifest_data, provisional_unclean_rows, make_loaders,
    make_model, set_seed, device_for_run, run_epoch, evaluate, summarise,
)

experiment_name = "unclean_as_labeled"
manifest, original_training_rows, validation_rows = manifest_data()
extra_rows = provisional_unclean_rows(manifest, original_training_rows, validation_rows)
assert len(extra_rows) == len({row["content_hash"] for row in extra_rows})
assert {row["content_hash"] for row in extra_rows}.isdisjoint(
    {row["content_hash"] for row in original_training_rows + validation_rows}
)
print("Original training:", len(original_training_rows))
print("Provisional unclean additions:", len(extra_rows))
print("Frozen validation:", len(validation_rows))
print("Unclean class counts:", dict(Counter(row["label"] for row in extra_rows)))


سلول بالا فقط فهرست را می‌سازد. `content_hash` از محتوای پیکسلی تصویر حساب شده است؛ اشتراک نداشتن hashها جلوی **تکرار دقیق** بین دادهٔ افزوده و validation را می‌گیرد، ولی تصاویر بسیار مشابه را تضمین نمی‌کند. `provisional` یعنی برچسب‌ها هنوز تک‌به‌تک تأیید نشده‌اند.

In [ ]:
import torch
from copy import deepcopy

set_seed(42)
experiment_device = device_for_run()
experiment_train_loader, experiment_val_loader, used_training_rows = make_loaders(
    manifest, original_training_rows, validation_rows, experiment_name
)
experiment_model, parameter_groups = make_model(experiment_name, experiment_device)
experiment_optimizer = torch.optim.Adam(parameter_groups)
experiment_epochs = 10
experiment_history = []
best_accuracy = -1.0
best_epoch = None
best_state = None

print("Device:", experiment_device)
print("Training images:", len(used_training_rows))
print("Validation images:", len(validation_rows))
print("Training batches per epoch:", len(experiment_train_loader))
print("Trainable parameters:", sum(p.numel() for p in experiment_model.parameters() if p.requires_grad))


در حلقهٔ بعدی، `run_epoch` وزن‌های مدل را با `backward()` و `optimizer.step()` تغییر می‌دهد. `evaluate` فقط روی validation اندازه‌گیری می‌کند و وزن‌ها را تغییر نمی‌دهد. پس از هر epoch، اگر دقت validation بهتر شود، یک کپی از وزن‌های همان لحظه در `best_state` نگه می‌داریم. این سلول حدود چند دقیقه زمان می‌برد؛ **فقط یک بار اجراش کن**.

In [ ]:
for epoch in range(1, experiment_epochs + 1):
    train_metrics = run_epoch(
        experiment_model, experiment_train_loader, experiment_optimizer,
        experiment_device, experiment_name
    )
    val_metrics = evaluate(
        experiment_model, experiment_val_loader, experiment_device,
        experiment_name
    )
    experiment_history.append({
        "epoch": epoch,
        "train": train_metrics,
        "validation": val_metrics,
    })
    if val_metrics["accuracy"] > best_accuracy:
        best_accuracy = val_metrics["accuracy"]
        best_epoch = epoch
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in experiment_model.state_dict().items()
        }
    print(
        f"Epoch {epoch:02d}/{experiment_epochs} | "
        f"train acc: {train_metrics['accuracy']:.1%} | "
        f"val acc: {val_metrics['accuracy']:.1%} | "
        f"val loss: {val_metrics['loss']:.4f}"
    )
print(f"Best validation accuracy: {best_accuracy:.1%} at epoch {best_epoch}")


سلول آخر بهترین وزن‌ها را برمی‌گرداند، معیارهای هشت کلاس را حساب می‌کند و فقط **آمار تجمیعی** را در گزارش ذخیره می‌کند. فایل checkpoint باینری و خصوصی است و در Git قرار نمی‌گیرد.

In [ ]:
import json

experiment_model.load_state_dict(best_state)
final_validation = evaluate(
    experiment_model, experiment_val_loader, experiment_device,
    experiment_name, with_predictions=True
)
experiment_summary = summarise(
    final_validation["truth"], final_validation["predictions"]
)
experiment_summary.update({
    "experiment": experiment_name,
    "split_fingerprint": manifest["fingerprint"],
    "training_images": len(used_training_rows),
    "validation_images": len(validation_rows),
    "best_epoch": best_epoch,
    "epochs": experiment_epochs,
    "history": experiment_history,
    "configuration": {
        "seed": 42, "batch_size": 32,
        "loss": "CrossEntropyLoss", "optimizer": "Adam", "learning_rate": 0.001,
        "unclean_policy": "known-class labels as supplied; provisional, not manually verified",
        "unclean_training_images": len(extra_rows),
    },
})
experiment_report_path = PROJECT_ROOT / "reports" / "experiment_results.json"
all_experiment_results = (
    json.loads(experiment_report_path.read_text())
    if experiment_report_path.exists() else {}
)
all_experiment_results[experiment_name] = experiment_summary
experiment_report_path.write_text(json.dumps(all_experiment_results, indent=2) + "\n")

experiment_checkpoint_path = PROJECT_ROOT / "checkpoints" / "unclean_as_labeled_best.pt"
experiment_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "architecture": "VehicleCNN",
    "strategy": experiment_name,
    "model_state_dict": best_state,
    "class_to_idx": manifest["class_to_idx"],
    "split_fingerprint": manifest["fingerprint"],
    "best_epoch": best_epoch,
    "validation_accuracy": experiment_summary["accuracy"],
    "preprocessing": "RGB pad 128; ToTensor; Normalize(0.5,0.5)",
}, experiment_checkpoint_path)
print("Validation accuracy:", f"{experiment_summary['accuracy']:.1%}")
print("Macro F1:", f"{experiment_summary['macro_f1']:.3f}")
print("Saved checkpoint:", experiment_checkpoint_path)


## نتیجهٔ آزمایش `unclean` و محدودیت مقایسه

با validation ثابتِ ۴۶۷ تصویر، مدل CNN پایه **۳۷۸/۴۶۷ = ۸۰٫۹٪** و مدل آموزش‌دیده با ۱۷۵۵ تصویر `unclean` موقت **۴۰۱/۴۶۷ = ۸۵٫۹٪** پاسخ درست داشت؛ بهبود **۲۳ تصویر یا حدود ۵ واحد درصد**. Macro-F1 نیز از **۰٫۸۰۹ به ۰٫۸۵۹** رسید. بهترین checkpoint اجرای جدید از **epoch 9** است، نه epoch 10.

اما برای `kamyun`، recall از **۶۳٫۳٪ به ۸۰٫۰٪** بهتر و precision از **۷۴٫۵٪ به ۶۴٫۰٪** بدتر شد. برای `kamyunet`، precision از **۶۰٫۷٪ به ۷۸٫۳٪** بهتر و recall از **۶۳٫۸٪ به ۶۲٫۱٪** اندکی بدتر شد. پس بالا رفتن دقت کلی به معنی بهترشدن همهٔ نوع خطاها نیست. ضعیف‌ترین precision اجرای جدید `kamyun` و ضعیف‌ترین recall آن `kamyunet` است.

این آزمایش یک مقایسهٔ مفید است، ولی دلیل قطعیِ بهبود را جدا نمی‌کند: با دادهٔ بیشتر، هر epoch هم batch و قدم‌های بهینه‌سازی بیشتری داشته است (۱۱۴ در برابر ۵۹). برچسب‌های `unclean` نیز تک‌به‌تک تأیید نشده‌اند. در پنج epoch نخستِ اجرای جدید، بهترین دقت **۸۲٫۹٪** بود؛ این تقریباً با بودجهٔ قدم‌های ۱۰ epoch مدل پایه قابل مقایسه است، اما جای آزمایش کاملاً کنترل‌شده را نمی‌گیرد. نتیجهٔ فعلی دلیل خوبی برای ادامهٔ بررسی `unclean` است، نه مجوز اصلاح خودکار برچسب‌ها.

## نیسان: سیاست برچسب و ممیزی پیش از آموزش

خروجی مسئله همچنان **۸ کلاس** دارد. مطابق توضیح منتورها، `neysan` را در زمان آموزش به `vanet` نگاشت می‌کنیم؛ روی دیسک هیچ تصویری جابه‌جا یا تغییرنام نمی‌شود. ۲۵۰ تصویر در مرور تصویریِ چندصفحه‌ای شبیه نیسان‌اند، اما نمونه‌های بسیار تاریک/ناقص و فریم‌های مشابه هم دیده می‌شوند. این مرور، تضمینِ بی‌خطا بودن همهٔ برچسب‌ها نیست.

ابتدا فهرست نیسان را از ممیزی قبلی می‌خوانیم، تصمیم‌های `exclude` را محترم می‌شماریم و وجود تکرار دقیق با آموزش و validation ثابت را می‌سنجیم. بخش ارزیابی نیسان جدا از validation هشت‌کلاسه است و برای انتخاب بهترین epoch استفاده نمی‌شود.


In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.run_experiments import (
    manifest_data, provisional_unclean_rows, make_model, set_seed,
    device_for_run, run_epoch, evaluate, summarise,
)

import csv
from collections import Counter

with (PROJECT_ROOT / "reports/data_review_decisions.csv").open(newline="", encoding="utf-8") as file:
    review_rows = list(csv.DictReader(file))

neysan_rows = [
    {"path": row["path"], "label": "vanet", "content_hash": row["content_hash"]}
    for row in review_rows
    if row["original_label"] == "neysan"
    and Path(row["path"]).parts[1] == "unclean"
    and row["decision"] != "exclude"
]
if any(row["original_label"] == "neysan" and row["decision"] == "relabel" for row in review_rows):
    raise ValueError("A Nissan relabel decision needs manual reconciliation before this experiment.")
if not neysan_rows or any(not row["content_hash"] for row in neysan_rows):
    raise ValueError("Nissan inventory is empty or contains an unreadable image; rerun the audit.")

manifest, original_training_rows, validation_rows = manifest_data()
known_unclean_rows = provisional_unclean_rows(manifest, original_training_rows, validation_rows)
existing_hashes = {row["content_hash"] for row in original_training_rows + validation_rows + known_unclean_rows}
assert len(neysan_rows) == len({row["content_hash"] for row in neysan_rows})
assert {row["content_hash"] for row in neysan_rows}.isdisjoint(existing_hashes)
print("Nissan candidates:", len(neysan_rows))
print("By source:", dict(Counter(row["path"].split("/")[0] for row in neysan_rows)))
print("Exact overlap with current training/validation:", 0)


تکرار دقیق کافی نیست: دو فریم از یک خودرو می‌توانند پیکسل‌های متفاوتی داشته باشند. در سلول بعد از **perceptual hash** استفاده می‌کنیم: هر تصویر به یک امضای ۶۴بیتی از الگوی کلی روشنایی تبدیل می‌شود؛ فاصلهٔ همینگ، تعداد بیت‌های متفاوت است. آستانهٔ ۱۲ یک قاعدهٔ احتیاطی است، نه اثبات هویت یکسان. نمونهٔ نزدیک به validation اصلی را موقتاً از این آزمایش کنار می‌گذاریم. نامزدهای نزدیک به آموزش موجود هم از بخش ارزیابی نیسان کنار می‌مانند. نمونه‌های نزدیک به هم در مجموعهٔ نیسان نیز در یک گروه می‌مانند تا یک گروه بین آموزش و ارزیابی شکسته نشود.


In [ ]:
import random
import numpy as np
from PIL import Image
from scipy.fft import dctn

def perceptual_hash(image_path):
    with Image.open(image_path) as image:
        gray = image.convert("L").resize((32, 32))
        pixels = np.asarray(gray, dtype=np.float64)
    low_frequencies = dctn(pixels, norm="ortho")[:8, :8]
    median = np.median(low_frequencies[1:, :])
    bits = (low_frequencies > median).ravel()
    return sum(int(bit) << index for index, bit in enumerate(bits))

NEAR_THRESHOLD = 12
validation_signatures = [perceptual_hash(PROJECT_ROOT / row["path"]) for row in validation_rows]
for row in neysan_rows:
    row["phash"] = perceptual_hash(PROJECT_ROOT / row["path"])

neysan_quarantine = [
    row for row in neysan_rows
    if min((row["phash"] ^ signature).bit_count() for signature in validation_signatures) <= NEAR_THRESHOLD
]
quarantined_paths = {row["path"] for row in neysan_quarantine}
neysan_eligible = [row for row in neysan_rows if row["path"] not in quarantined_paths]
print("Near existing validation; quarantined:", len(neysan_quarantine))
print("Eligible for this experiment:", len(neysan_eligible))


In [ ]:
# Union-find: group Nissan images with perceptual-hash distance <= 12.
parent = list(range(len(neysan_eligible)))

def find_group(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index

for left in range(len(neysan_eligible)):
    for right in range(left + 1, len(neysan_eligible)):
        distance = (neysan_eligible[left]["phash"] ^ neysan_eligible[right]["phash"]).bit_count()
        if distance <= NEAR_THRESHOLD:
            parent[find_group(left)] = find_group(right)

groups = {}
for index, row in enumerate(neysan_eligible):
    groups.setdefault(find_group(index), []).append(row)

# Protect the Nissan holdout from near matches already present in training.
existing_train_signatures = [
    perceptual_hash(PROJECT_ROOT / row["path"])
    for row in original_training_rows + known_unclean_rows
]

def clear_of_existing_training(group):
    return all(
        min((row["phash"] ^ signature).bit_count() for signature in existing_train_signatures)
        > NEAR_THRESHOLD
        for row in group
    )

# A mixed-source group stays in training. Select about 20% per source for holdout.
rng = random.Random(42)
targets = {source: round(0.2 * count) for source, count in Counter(
    row["path"].split("/")[0] for row in neysan_eligible
).items()}
neysan_holdout_rows = []
for source, target in sorted(targets.items()):
    source_groups = [group for group in groups.values() if {
        row["path"].split("/")[0] for row in group
    } == {source} and clear_of_existing_training(group)]
    source_groups.sort(key=lambda group: group[0]["path"])
    rng.shuffle(source_groups)
    selected = 0
    for group in source_groups:
        if selected + len(group) <= target:
            neysan_holdout_rows.extend(group)
            selected += len(group)
    if selected != target:
        raise ValueError(f"Could not create a {target}-image holdout for {source} without splitting a group")

holdout_paths = {row["path"] for row in neysan_holdout_rows}
neysan_training_rows = [row for row in neysan_eligible if row["path"] not in holdout_paths]
assert len(neysan_training_rows) + len(neysan_holdout_rows) + len(neysan_quarantine) == len(neysan_rows)
assert {row["content_hash"] for row in neysan_training_rows}.isdisjoint(
    {row["content_hash"] for row in neysan_holdout_rows}
)
assert all(
    (train["phash"] ^ holdout["phash"]).bit_count() > NEAR_THRESHOLD
    for train in neysan_training_rows for holdout in neysan_holdout_rows
)
assert all(clear_of_existing_training([row]) for row in neysan_holdout_rows)
print("Nissan training:", len(neysan_training_rows))
print("Nissan holdout:", len(neysan_holdout_rows))
print("Holdout by source:", dict(Counter(row["path"].split("/")[0] for row in neysan_holdout_rows)))
print("Perceptual groups:", len(groups))


`neysan_training_rows` به `vanet` برچسب خورده و فقط وارد آموزش می‌شود. `neysan_holdout_rows` در آموزش و انتخاب بهترین epoch شرکت نمی‌کند. فایل تقسیم‌بندی خصوصی ذخیره می‌شود تا آزمایش قابل تکرار باشد؛ `fingerprint` تقسیم اصلی و validation ثابت تغییر نمی‌کند. آستانهٔ perceptual hash محافظه‌کارانه است، ولی نمی‌تواند تمام فریم‌های مشابه را پیدا کند؛ نتیجهٔ holdout را نباید معادل تست پنهان منتورها دانست.


In [ ]:
import hashlib
import json

neysan_split = {
    "seed": 42,
    "label_policy": "neysan -> vanet (8-class classifier)",
    "near_hash_threshold": NEAR_THRESHOLD,
    "base_split_fingerprint": manifest["fingerprint"],
    "training_paths": sorted(row["path"] for row in neysan_training_rows),
    "holdout_paths": sorted(row["path"] for row in neysan_holdout_rows),
    "quarantined_paths": sorted(row["path"] for row in neysan_quarantine),
}
neysan_split["fingerprint"] = hashlib.sha256(
    json.dumps(neysan_split, sort_keys=True).encode("utf-8")
).hexdigest()
private_split_path = PROJECT_ROOT / "reports/neysan_split.json"
private_split_path.write_text(json.dumps(neysan_split, indent=2) + "\n", encoding="utf-8")
summary_path = PROJECT_ROOT / "reports/data_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary.update({
    "neysan_target_label": "vanet",
    "neysan_near_validation_quarantined": len(neysan_quarantine),
    "neysan_training_images": len(neysan_training_rows),
    "neysan_holdout_images": len(neysan_holdout_rows),
})
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Saved private split:", private_split_path.name)
print("Split fingerprint:", neysan_split["fingerprint"][:12])


## آزمایش CNN با نیسان

برای اینکه اثر تصمیم نیسان دیده شود، مدل CNN را با همان معماری، seed، optimizer و ۱۰ epoch آزمایش قبلی **از ابتدا** آموزش می‌دهیم. آموزش شامل train اصلی، همان ۱۷۵۵ تصویر unclean هشت‌کلاسه و نیسان‌های انتخاب‌شده با برچسب `vanet` است. validation اصلیِ ۴۶۷ تصویری برای انتخاب بهترین epoch ثابت می‌ماند؛ holdout نیسان فقط در انتها سنجیده می‌شود. تفاوت این اجرا با آزمایش قبل فقط تعداد تصاویر نیسان نیست: در هر epoch چند قدم optimizer بیشتر اجرا می‌شود، پس تغییر دقت را با احتیاط تفسیر می‌کنیم.


In [ ]:
from torch.utils.data import DataLoader
from scripts.run_experiments import VehicleDataset, cnn_transform

experiment_name = "unclean_plus_neysan"
used_training_rows = original_training_rows + known_unclean_rows + neysan_training_rows
training_hashes = {row["content_hash"] for row in used_training_rows}
validation_hashes = {row["content_hash"] for row in validation_rows}
holdout_hashes = {row["content_hash"] for row in neysan_holdout_rows}
assert len(training_hashes) == len(used_training_rows)
assert training_hashes.isdisjoint(validation_hashes | holdout_hashes)
assert validation_hashes.isdisjoint(holdout_hashes)

train_loader = DataLoader(
    VehicleDataset(used_training_rows, cnn_transform(), manifest["class_to_idx"]),
    batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(42), num_workers=0,
)
val_loader = DataLoader(
    VehicleDataset(validation_rows, cnn_transform(), manifest["class_to_idx"]),
    batch_size=32, shuffle=False, num_workers=0,
)
neysan_holdout_loader = DataLoader(
    VehicleDataset(neysan_holdout_rows, cnn_transform(), manifest["class_to_idx"]),
    batch_size=32, shuffle=False, num_workers=0,
)
set_seed(42)
experiment_device = device_for_run()
experiment_model, parameter_groups = make_model(experiment_name, experiment_device)
experiment_optimizer = torch.optim.Adam(parameter_groups)
experiment_epochs = 30
experiment_history = []
best_accuracy, best_epoch, best_state = -1.0, None, None
print("Train / fixed validation / Nissan holdout:",
      len(used_training_rows), len(validation_rows), len(neysan_holdout_rows))
print("Training images by class:", dict(Counter(row["label"] for row in used_training_rows)))
print("Batches per epoch:", len(train_loader), "Device:", experiment_device)


حالا سلول آموزش را **یک بار** اجرا کن. `run_epoch` از روی batchها loss و gradient را حساب می‌کند و وزن‌ها را با `optimizer.step()` به‌روز می‌کند؛ `evaluate` فقط اندازه‌گیری می‌کند. وزن بهترین epoch بر اساس validation ثابت کپی می‌شود. به holdout نیسان تا پایان این حلقه نگاه نمی‌کنیم.


In [ ]:
for epoch in range(1, experiment_epochs + 1):
    train_metrics = run_epoch(
        experiment_model, train_loader, experiment_optimizer, experiment_device, experiment_name
    )
    val_metrics = evaluate(experiment_model, val_loader, experiment_device, experiment_name)
    experiment_history.append({"epoch": epoch, "train": train_metrics, "validation": val_metrics})
    if val_metrics["accuracy"] > best_accuracy:
        best_accuracy = val_metrics["accuracy"]
        best_epoch = epoch
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in experiment_model.state_dict().items()
        }
    print(f"Epoch {epoch:02d}/{experiment_epochs} | "
          f"train acc: {train_metrics['accuracy']:.1%} | "
          f"val acc: {val_metrics['accuracy']:.1%} | "
          f"val loss: {val_metrics['loss']:.4f}")
print(f"Best fixed-validation accuracy: {best_accuracy:.1%} at epoch {best_epoch}")


در انتها وزن بهترین epoch را برمی‌گردانیم و **دو مجموعهٔ متفاوت** را گزارش می‌کنیم: دقت/Macro-F1 روی validation هشت‌کلاسه، و نسبت نیسان‌هایی که به `vanet` نسبت داده شده‌اند روی holdout نیسان. چون تمام مثال‌های holdout از یک کلاسِ مورد انتظارند، عدد دوم فقط recall نیسان برای برچسب وانت است، نه دقت کلی هشت کلاس و نه precision وانت. confidenceِ softmax را نیز توصیفی می‌خوانیم؛ به‌تنهایی احتمالِ کالیبره‌شده یا معیار مطمئنِ ناشناخته بودن نیست.


In [ ]:
experiment_model.load_state_dict(best_state)
final_validation = evaluate(
    experiment_model, val_loader, experiment_device, experiment_name, with_predictions=True
)
final_neysan = evaluate(
    experiment_model, neysan_holdout_loader, experiment_device, experiment_name,
    with_predictions=True,
)
experiment_summary = summarise(final_validation["truth"], final_validation["predictions"])
vanet_index = manifest["class_to_idx"]["vanet"]
neysan_correct = sum(label == vanet_index for label in final_neysan["predictions"])
assert all(label == vanet_index for label in final_neysan["truth"])
experiment_summary.update({
    "experiment": experiment_name,
    "split_fingerprint": manifest["fingerprint"],
    "neysan_split_fingerprint": neysan_split["fingerprint"],
    "training_images": len(used_training_rows),
    "validation_images": len(validation_rows),
    "neysan_holdout_images": len(neysan_holdout_rows),
    "neysan_holdout_correct_as_vanet": neysan_correct,
    "neysan_holdout_vanet_recall": neysan_correct / len(neysan_holdout_rows),
    "neysan_holdout_mean_confidence": sum(final_neysan["confidences"]) / len(neysan_holdout_rows),
    "best_epoch": best_epoch,
    "epochs": len(experiment_history),
    "history": experiment_history,
    "configuration": {
        "seed": 42, "batch_size": 32, "loss": "CrossEntropyLoss", "optimizer": "Adam",
        "learning_rate": 0.001, "unclean_policy": "known classes as supplied (provisional); neysan -> vanet",
        "known_unclean_training_images": len(known_unclean_rows),
        "neysan_training_images": len(neysan_training_rows),
        "neysan_near_validation_quarantined": len(neysan_quarantine),
    },
})
report_path = PROJECT_ROOT / "reports/experiment_results.json"
all_results = json.loads(report_path.read_text()) if report_path.exists() else {}
all_results[experiment_name] = experiment_summary
report_path.write_text(json.dumps(all_results, indent=2) + "\n")
checkpoint_path = PROJECT_ROOT / "checkpoints/unclean_plus_neysan_best.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "architecture": "VehicleCNN", "strategy": experiment_name,
    "model_state_dict": best_state, "class_to_idx": manifest["class_to_idx"],
    "split_fingerprint": manifest["fingerprint"],
    "neysan_split_fingerprint": neysan_split["fingerprint"],
    "best_epoch": best_epoch, "validation_accuracy": experiment_summary["accuracy"],
    "preprocessing": "RGB pad 128; ToTensor; Normalize(0.5,0.5)",
}, checkpoint_path)
print("Fixed validation:", f"{experiment_summary['accuracy']:.1%}",
      "Macro-F1:", f"{experiment_summary['macro_f1']:.3f}")
print("Nissan holdout predicted as vanet:",
      f"{neysan_correct}/{len(neysan_holdout_rows)} = {neysan_correct / len(neysan_holdout_rows):.1%}")
print("Checkpoint:", checkpoint_path.name)


## نتیجهٔ اجرای طولانی‌تر CNN و آماده‌سازی قدم بعد

در خروجی ذخیره‌شدهٔ بالا، `range(1, 30)` **۲۹ epoch** اجرا کرده است؛ عبارت `/10` در خروجی قدیمی فقط از متغیرِ به‌روزرسانی‌نشده آمده بود. بهترین دقت validation ثابت در epoch ۲۰ (و با دقت مساوی در epoch ۲۴) **۴۱۵/۴۶۷ = ۸۸٫۹٪** و Macro-F1 حدود **۰٫۸۹۰** شد. بهترین وزن‌ها با شرط `>` از epoch ۲۰ ذخیره شدند. تشخیص نیسان در holdout همچنان **۴۳/۴۷ = ۹۱٫۵٪** است.

این نتیجه را نمی‌توان فقط به افزودن نیسان نسبت داد: اجرای قبلی ۱۰ epoch داشت، ولی این اجرا ۲۹ epoch. دقت آموزش از epoch ۱۱ به ۱۰۰٪ رسید، در حالی که validation loss از حدود ۰٫۵۱ در epoch ۴ به حدود ۰٫۷۰ در epoch ۲۰ رسید؛ بنابراین افزایش دقت با نشانه‌هایی از اطمینان زیاد روی برخی خطاها همراه است. برای اجرای دقیق ۳۰ epoch در آینده، ابتدا سلول ساخت مدل و optimizer را دوباره اجرا کن تا وزن‌ها، optimizer و history از ابتدا ساخته شوند؛ اجرای دوبارهٔ حلقه به‌تنهایی آموزش قبلی را ادامه می‌دهد. کد بالا اکنون `experiment_epochs = 30` و `range(1, experiment_epochs + 1)` دارد، اما **خروجیِ ذخیره‌شدهٔ فعلی متعلق به اجرای ۲۹-epoch است**.


## ResNet18، قدم ۱: معماری مدل ازپیش‌آموزش‌دیده

طبق صورت پروژه، باید یک ResNet18 با وزن‌های آموزش‌دیده روی ImageNet را برای **استخراج ویژگی** و سپس **fine-tuning لایهٔ `layer4`** مقایسه کنیم. فعلاً فقط مدل را می‌سازیم و لایهٔ ورودی و لایهٔ آخرش را می‌بینیم؛ این سلول **هیچ آموزشی انجام نمی‌دهد** و وزن‌ها را تغییر نمی‌دهد. فایل وزن ImageNet در کش محلی موجود است.

سؤال کلیدی: لایهٔ `fc` فعلی ۵۱۲ ویژگی ورودی می‌گیرد و ۱۰۰۰ score می‌دهد. این ۱۰۰۰ خروجی برای مسئلهٔ ما نیستند؛ در قدم بعد سرِ مدل را با یک لایهٔ ۸خروجی جایگزین می‌کنیم. طبق صورت پروژه، `conv1` و `maxpool` اصلی را برای مدل pretrained دست‌نخورده می‌گذاریم و از preprocessing سازگار با ImageNet استفاده می‌کنیم.


In [ ]:
import json
from pathlib import Path
from torchvision import models

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
resnet_manifest = json.loads((project_root / "reports/base_split.json").read_text())

resnet_weights = models.ResNet18_Weights.DEFAULT
resnet18_model = models.resnet18(weights=resnet_weights)

print("First convolution:", resnet18_model.conv1)
print("Last layer:", resnet18_model.fc)
print("Features entering fc:", resnet18_model.fc.in_features)
print("Original output scores:", resnet18_model.fc.out_features)
print("Our vehicle classes:", len(resnet_manifest["class_to_idx"]))
print("ImageNet preprocessing:", resnet_weights.transforms())


## ResNet18، قدم ۲: backbone ثابت، سرِ جدید قابل‌آموزش

ترتیب این سه کار مهم است و از فایل تدریس استاد پیروی می‌کند: **(۱) بارگذاری وزن pretrained، (۲) فریز کردن پارامترهای موجود، (۳) جایگزین کردن `fc` با یک لایهٔ تازهٔ ۸خروجی**. اگر لایهٔ تازه را قبل از فریز کردن بسازیم، آن هم فریز می‌شود. لایهٔ تازه به‌صورت پیش‌فرض `requires_grad=True` دارد.

در این مرحله فقط مدل را آماده می‌کنیم؛ هنوز هیچ batch، loss، `backward()` یا `optimizer.step()` نداریم. backbone در forward همچنان ویژگی تولید می‌کند، اما وزن‌هایش در آموزشِ مرحلهٔ اول تغییر نمی‌کنند. انتظار داریم تعداد پارامترهای قابل‌آموزش فقط `512 × 8 + 8 = 4104` باشد: برای هر یک از ۸ کلاس، ۵۱۲ وزن و یک bias. `eval()` کار فریز کردن را انجام نمی‌دهد؛ رفتار BatchNorm را در قدم‌های بعد جداگانه تنظیم می‌کنیم.


In [ ]:
from torch import nn

# Same pretrained architecture as the inspection cell above.
feature_model = models.resnet18(weights=resnet_weights)

# Freeze every parameter that came with the pretrained model.
feature_model.requires_grad_(False)

num_features = feature_model.fc.in_features
num_classes = len(resnet_manifest["class_to_idx"])
feature_model.fc = nn.Linear(num_features, num_classes)

trainable_names = [
    name for name, parameter in feature_model.named_parameters()
    if parameter.requires_grad
]
total_parameters = sum(parameter.numel() for parameter in feature_model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in feature_model.parameters()
    if parameter.requires_grad
)

assert trainable_names == ["fc.weight", "fc.bias"]
assert trainable_parameters == num_features * num_classes + num_classes
print("New classifier:", feature_model.fc)
print("Backbone conv1 trainable:", feature_model.conv1.weight.requires_grad)
print("Classifier weight trainable:", feature_model.fc.weight.requires_grad)
print("Trainable parameter names:", trainable_names)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)


## ResNet18، قدم ۳: تصویر را برای وزن‌های ImageNet آماده کنیم

در CNN خودمان تصویر را با padding به `128×128` رساندیم و با میانگین/انحراف معیار `0.5` نرمال کردیم. وزن‌های آمادهٔ ResNet18 با روش دیگری آموزش دیده‌اند؛ بنابراین آن pipeline را بدون بررسی برای ResNet تکرار نمی‌کنیم. دستور `resnet_weights.transforms()` نسخهٔ ارزیابیِ سازگار با همین وزن‌ها را می‌دهد: ضلع کوتاه تصویر به ۲۵۶ می‌رسد، مرکز آن به `224×224` بریده می‌شود، پیکسل‌ها tensor می‌شوند و هر کانال با آمار ImageNet نرمال می‌شود.

سلول زیر یک **عکس از بخش آموزش** را قبل و بعد از `Resize → CenterCrop` نشان می‌دهد و شکل tensor نهایی را چاپ می‌کند. عکس نمایش‌داده‌شده دادهٔ خصوصی است و خروجی تصویری نوت‌بوک وارد Git نمی‌شود. هنوز مدل را آموزش یا ارزیابی نمی‌کنیم. به این دقت کن که بریدن مرکز تصویر ممکن است در عکس‌های باریک بخشی از خودرو را حذف کند؛ بعد از دیدن نمونه دربارهٔ روش مناسب دادهٔ خودمان تصمیم می‌گیریم.


In [ ]:
from PIL import Image
from IPython.display import display
from torchvision import transforms
from torchvision.transforms import InterpolationMode

resnet_eval_transform = resnet_weights.transforms()

# Choose a tall training image so any center-crop loss is easy to see.
sample_image_path = None
for relative_path in resnet_manifest["training_paths"]:
    candidate = project_root / relative_path
    with Image.open(candidate) as image:
        width, height = image.size
    if max(width, height) / min(width, height) >= 1.8:
        sample_image_path = candidate
        break
if sample_image_path is None:
    sample_image_path = project_root / resnet_manifest["training_paths"][0]

with Image.open(sample_image_path) as image:
    original_image = image.convert("RGB")

# For display only: the geometric part before ToTensor and Normalize.
geometry_only = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
])
preview_image = geometry_only(original_image)
resnet_input = resnet_eval_transform(original_image)

print("Original size (width, height):", original_image.size)
print("After resize and center crop:", preview_image.size)
print("Tensor shape (channels, height, width):", tuple(resnet_input.shape))
print("Tensor value range after ImageNet normalization:",
      round(resnet_input.min().item(), 2), round(resnet_input.max().item(), 2))
print("Original image, then model-visible crop:")
display(original_image, preview_image)


## ResNet18، قدم ۴: transform سادهٔ درس استاد

برای مسیر اصلی، آزمایش جانبیِ padding را فعلاً کنار می‌گذاریم. در کد آموزشی استاد، تصویر به‌صورت مستقیم به `224×224` تغییر اندازه داده می‌شود، به tensor تبدیل می‌شود و با میانگین و انحراف معیار ImageNet نرمال می‌شود. این روش تمام تصویر را نگه می‌دارد و center crop ندارد؛ در عکس‌های کشیده ممکن است نسبت طول‌به‌عرض را تغییر دهد. فعلاً همین روش ساده را برای آموزش و validation استفاده می‌کنیم تا feature extraction و سپس fine-tuning را یاد بگیریم. augmentation و مقایسهٔ روش‌های دیگر را بعداً به‌صورت آزمایش جدا انجام می‌دهیم.

سلول بعد فقط transform را تعریف می‌کند و روی همان یک عکس آموزشی شکل tensor را بررسی می‌کند. وزن‌های مدل تغییر نمی‌کنند و هنوز DataLoader یا حلقهٔ آموزش نداریم.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transformed_sample = resnet_transform(original_image)
print("Original size (width, height):", original_image.size)
print("ResNet tensor shape (channels, height, width):", tuple(transformed_sample.shape))
print("Same normalization as pretrained weights:",
      IMAGENET_MEAN == list(resnet_eval_transform.mean)
      and IMAGENET_STD == list(resnet_eval_transform.std))


## ResNet18، قدم ۵: همان فهرست دادهٔ آزمایش قبلی را بسازیم

برای مقایسهٔ CNN و ResNet، فعلاً داده را عوض نمی‌کنیم. فهرست آموزش از سه بخش تشکیل می‌شود: (۱) دادهٔ اصلیِ تقسیم ثابت، (۲) `unclean` هشت‌کلاسهٔ آزمایش قبلی، (۳) نیسان‌های انتخاب‌شده با برچسب هدف `vanet`. validation هشت‌کلاسه و holdout نیسان جدا باقی می‌مانند. این سلول فقط **مسیر، برچسب و hash** عکس‌ها را در حافظه جمع می‌کند؛ نه تصویر را بارگذاری می‌کند، نه مدلی را آموزش می‌دهد، نه فایل خام را جابه‌جا می‌کند.

وجود `assert`ها مهم است: اگر تقسیم خصوصی نیسان متعلق به تقسیم اصلی دیگری باشد یا یک محتوای دقیق در دو مجموعه ظاهر شود، قبل از آموزش خطا می‌گیریم. این بررسی تکرار دقیق است؛ بررسی شباهت تصویری در مرحلهٔ ممیزی نیسان انجام شده است.


In [ ]:
import csv
import sys
from collections import Counter

sys.path.insert(0, str(project_root))
from scripts.run_experiments import manifest_data, provisional_unclean_rows

resnet_manifest, base_training_rows, resnet_validation_rows = manifest_data()
known_unclean_rows = provisional_unclean_rows(
    resnet_manifest, base_training_rows, resnet_validation_rows
)
neysan_split = json.loads((project_root / "reports/neysan_split.json").read_text())
assert neysan_split["base_split_fingerprint"] == resnet_manifest["fingerprint"]

with (project_root / "reports/data_review_decisions.csv").open(newline="", encoding="utf-8") as file:
    content_hash_by_path = {
        row["path"]: row["content_hash"]
        for row in csv.DictReader(file)
        if row["original_label"] == "neysan"
    }

def map_neysan_to_vanet(paths):
    return [
        {"path": path, "label": "vanet", "content_hash": content_hash_by_path[path]}
        for path in paths
    ]

neysan_training_rows = map_neysan_to_vanet(neysan_split["training_paths"])
resnet_neysan_holdout_rows = map_neysan_to_vanet(neysan_split["holdout_paths"])
resnet_training_rows = base_training_rows + known_unclean_rows + neysan_training_rows

training_hashes = {row["content_hash"] for row in resnet_training_rows}
validation_hashes = {row["content_hash"] for row in resnet_validation_rows}
holdout_hashes = {row["content_hash"] for row in resnet_neysan_holdout_rows}
assert len(training_hashes) == len(resnet_training_rows)
assert training_hashes.isdisjoint(validation_hashes | holdout_hashes)
assert validation_hashes.isdisjoint(holdout_hashes)

print("Base training:", len(base_training_rows))
print("Known-class unclean additions:", len(known_unclean_rows))
print("Nissan -> vanet additions:", len(neysan_training_rows))
print("Total ResNet training:", len(resnet_training_rows))
print("Fixed validation:", len(resnet_validation_rows))
print("Nissan holdout:", len(resnet_neysan_holdout_rows))
print("Training counts by class:", dict(Counter(row["label"] for row in resnet_training_rows)))


## ResNet18، قدم ۶: Dataset و DataLoader

`VehicleDataset` به هر **index** یک مسیر و برچسب مرتبط می‌کند؛ وقتی DataLoader آن index را بخواهد، تصویر از فایل خوانده می‌شود، `resnet_transform` روی آن اجرا می‌شود و `(image_tensor, label_index)` برمی‌گردد. پس فهرست‌های قدم قبل هنوز خودِ عکس‌ها نبودند. DataLoader نمونه‌ها را در batchهای ۳۲تایی جمع می‌کند.

آموزش را با `shuffle=True` می‌خوانیم تا ترتیب نمونه‌ها در هر epoch عوض شود؛ seed آن ۴۲ است. validation و holdout با `shuffle=False` می‌مانند تا ترتیب پیش‌بینی‌ها قابل پیگیری باشد. `num_workers=0` یعنی همین فرایند نوت‌بوک تصاویر را بارگذاری می‌کند؛ برای شروع و دیباگ ساده‌تر است. تمام این مجموعه‌ها از transform سادهٔ قدم ۴ استفاده می‌کنند. در این سلول فقط **اولین batch آموزش** را باز می‌کنیم؛ وزن‌های مدل تغییری نمی‌کنند.


In [ ]:
import torch
from torch.utils.data import DataLoader
from scripts.run_experiments import VehicleDataset

resnet_train_dataset = VehicleDataset(
    resnet_training_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_val_dataset = VehicleDataset(
    resnet_validation_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_neysan_dataset = VehicleDataset(
    resnet_neysan_holdout_rows, resnet_transform, resnet_manifest["class_to_idx"]
)

resnet_train_loader = DataLoader(
    resnet_train_dataset, batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(42), num_workers=0,
)
resnet_val_loader = DataLoader(
    resnet_val_dataset, batch_size=32, shuffle=False, num_workers=0,
)
resnet_neysan_loader = DataLoader(
    resnet_neysan_dataset, batch_size=32, shuffle=False, num_workers=0,
)

batch_images, batch_labels = next(iter(resnet_train_loader))
print("Dataset lengths:", len(resnet_train_dataset), len(resnet_val_dataset), len(resnet_neysan_dataset))
print("Batches per epoch:", len(resnet_train_loader))
print("Images shape [batch, RGB, height, width]:", tuple(batch_images.shape))
print("Labels shape [batch]:", tuple(batch_labels.shape))
print("First five label indices:", batch_labels[:5].tolist())


## ResNet18، قدم ۷: یک forward pass و محاسبهٔ loss

ابتدا مدل و یک batch را روی یک device می‌گذاریم. برای آنکه آمار داخلی BatchNormِ backbone ثابت بماند، کل مدل را در حالت `eval()` می‌گذاریم و سپس head را با `fc.train()` در حالت آموزش می‌گذاریم؛ این حالت‌ها **فریزِ پارامتر نیستند**. پارامترهای backbone قبلاً با `requires_grad_(False)` فریز شدند و سرِ تازه قابل‌آموزش است. از `torch.no_grad()` استفاده نمی‌کنیم، چون در قدم بعد می‌خواهیم گرادیانِ سر را با `backward()` حساب کنیم.

خروجی `feature_model(images)` باید برای ۳۲ عکس، شکل `[32, 8]` داشته باشد: در هر ردیف هشت **logit/score خام**، نه هشت احتمال. `CrossEntropyLoss` این scoreها و شمارهٔ کلاس درست را می‌گیرد؛ نباید پیش از آن خودمان `softmax` بزنیم. این سلول فقط forward و loss را حساب می‌کند، هیچ وزنی را تغییر نمی‌دهد.


In [ ]:
from scripts.run_experiments import device_for_run

resnet_device = device_for_run()
feature_model = feature_model.to(resnet_device)
feature_model.eval()       # Keep frozen backbone BatchNorm statistics fixed.
feature_model.fc.train()   # The new classifier is the training part.

images_for_resnet = batch_images.to(resnet_device)
labels_for_resnet = batch_labels.to(resnet_device)
class_scores = feature_model(images_for_resnet)
loss_fn = nn.CrossEntropyLoss()
batch_loss = loss_fn(class_scores, labels_for_resnet)

assert class_scores.shape == (len(batch_labels), len(resnet_manifest["class_to_idx"]))
print("Device:", resnet_device)
print("Scores shape [batch, classes]:", tuple(class_scores.shape))
print("One image's eight raw scores:",
      [round(value, 3) for value in class_scores[0].detach().cpu().tolist()])
print("That image's true label index:", labels_for_resnet[0].item())
print("Batch loss:", round(batch_loss.item(), 4))
print("Scores require gradient:", class_scores.requires_grad)
print("Backbone training mode:", feature_model.layer4.training)
print("Classifier training mode:", feature_model.fc.training)


## ResNet18، قدم ۸: گرادیانِ سرِ تازه را بررسی کنیم

`backward()` مشتق loss را برای پارامترهای قابل‌آموزش حساب می‌کند؛ **به‌تنهایی وزن‌ها را تغییر نمی‌دهد**. ابتدا `zero_grad(set_to_none=True)` گرادیان باقی‌مانده از اجرای قبلی سلول را پاک می‌کند. سپس forward و loss را **دوباره** می‌سازیم تا اگر این سلول را تکرار کردی، graph تازه‌ای برای `backward()` داشته باشیم. در پایان باید `.grad` در `conv1` و `layer4` برابر `None` باشد، ولی در `fc.weight` و `fc.bias` وجود داشته باشد. شکل گرادیان `fc.weight` باید `[8, 512]` باشد: به ازای هر کلاس، مشتقِ ۵۱۲ وزن.

این سلول هنوز optimizer و `optimizer.step()` ندارد؛ پس حتی با وجود گرادیانِ `fc`، وزن‌ها همان مقادیر قبلی می‌مانند. این دقیقاً تفاوتِ «محاسبهٔ جهت تغییر» و «اعمال تغییر» است.


In [ ]:
feature_model.eval()
feature_model.fc.train()
feature_model.zero_grad(set_to_none=True)

head_weights_before = feature_model.fc.weight.detach().clone()
fresh_scores = feature_model(images_for_resnet)
fresh_loss = loss_fn(fresh_scores, labels_for_resnet)
fresh_loss.backward()

assert feature_model.conv1.weight.grad is None
assert feature_model.layer4[0].conv1.weight.grad is None
assert feature_model.fc.weight.grad is not None
assert feature_model.fc.bias.grad is not None

print("Conv1 requires_grad:", feature_model.conv1.weight.requires_grad)
print("Conv1 gradient:", feature_model.conv1.weight.grad)
print("Layer4 gradient:", feature_model.layer4[0].conv1.weight.grad)
print("FC requires_grad:", feature_model.fc.weight.requires_grad)
print("FC weight gradient shape:", tuple(feature_model.fc.weight.grad.shape))
print("FC bias gradient shape:", tuple(feature_model.fc.bias.grad.shape))
print("FC gradient norm:", round(feature_model.fc.weight.grad.norm().item(), 4))
print("FC weights unchanged after backward:",
      torch.equal(head_weights_before, feature_model.fc.weight.detach()))
